# Sistem Deteksi Kesiapan Panen Padi

**Drone → GCS → Stitching ODM → Orthophoto → Klasifikasi HSV 2-Kelas → Visualisasi GIS**

## Tentang Pipeline Ini

1. **Drone** mengambil puluhan foto overlap di atas sawah
2. Foto ditransfer ke **GCS / laptop**
3. Foto distitch menjadi **satu orthophoto** menggunakan **OpenDroneMap (ODM)** yang berjalan **offline** di Docker
4. Orthophoto diklasifikasi dengan **segmentasi HSV 2 kelas**: **Siap Panen** atau **Belum Siap Panen**
5. Hasil bisa divisualisasikan di **QGIS** dan **website**

### Disclaimer Akademis

> ⚠️ Sistem mendeteksi kesiapan panen berdasarkan **indikator visual warna** dari orthophoto UAV, **bukan** pengukuran parameter fisiologis langsung (kadar air, berat butir gabah). Hasil dimaksudkan sebagai pendukung keputusan dan tidak menggantikan inspeksi lapangan langsung oleh petani atau penyuluh pertanian.

### Prasyarat Sebelum Menjalankan

1. **Docker Desktop** terpasang & berjalan (`docker --version` di terminal harus menjawab)
2. **NodeODM container** sudah dijalankan dengan `start_nodeodm.bat` (atau perintah `docker run -d -p 3000:3000 --name nodeodm opendronemap/nodeodm`)
3. **Foto drone** Anda sudah diletakkan di folder `data/drone_images/` dengan format JPG dan **metadata EXIF GPS** masih ada
4. **Library Python** terpasang (jalankan `setup_environment.bat` sekali)


---

## Sel 1 — Instalasi Library

In [ ]:
# Jalankan sekali
%pip install --quiet pyodm opencv-python numpy matplotlib rasterio shapely Pillow pyproj exifread scikit-image tqdm pandas requests sqlmodel

print("✅ Instalasi library selesai. Lanjut ke Sel 2.")

## Sel 2 — Import Library

In [ ]:
# ============================================================================
# Import Library
# ============================================================================
import os
import sys
import json
import time
import shutil
import warnings
import subprocess
from datetime import datetime
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image

import rasterio
from rasterio.plot import reshape_as_image
import pandas as pd
from tqdm.auto import tqdm

# Library khusus PyODM untuk berbicara ke NodeODM
from pyodm import Node
from pyodm.exceptions import NodeConnectionError, NodeServerError, TaskFailedError

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("✅ Semua library berhasil dimuat.")
print(f"   Python    : {sys.version.split()[0]}")
print(f"   NumPy     : {np.__version__}")
print(f"   OpenCV    : {cv2.__version__}")
print(f"   Rasterio  : {rasterio.__version__}")
print(f"   Pandas    : {pd.__version__}")
import pyodm
print(f"   PyODM     : {pyodm.__version__ if hasattr(pyodm, '__version__') else '(terpasang)'}")

## Sel 3 — Panel Konfigurasi (CONFIG)

**`odm_options`** — parameter ODM saat stitching. Default sudah dioptimalkan untuk laptop dengan RAM 8 GB:
- `orthophoto-resolution: 5` cm/piksel → balance kualitas vs waktu (turunkan ke 3 untuk lebih detail, naikkan ke 10 untuk lebih cepat)
- `feature-quality: medium` → trade-off speed vs accuracy (`low` untuk laptop spek rendah, `high` untuk hasil terbaik)
- `fast-orthophoto: true` → skip 3D model, hanya hasilkan orthophoto 2D (mempercepat 2–4×)

**`threshold_siap_panen`** — persentase minimum piksel kuning agar lahan diklasifikasi "Siap Panen". Default `0.60` (60%).

**`hsv_*`** — skala OpenCV (H: 0–179, S/V: 0–255).

In [ ]:
# ============================================================================
# REGISTRY LAHAN — ganti SATU angka saja saat pindah lahan
# ----------------------------------------------------------------------------
# Ubah LAHAN_AKTIF, lalu Restart Kernel dan Run All. Folder foto drone dan
# nama lahan yang masuk ke database ikut berubah otomatis, sehingga tidak
# mungkin lagi foto lahan 2 tersimpan dengan label "Lahan 1".
# CATATAN: sesuaikan "folder" di bawah dengan struktur foldermu yang asli.
# ============================================================================
LAHAN = {
    1: {"folder": "data/drone_images/1", "nama": "Lahan 1"},
    2: {"folder": "data/drone_images/2", "nama": "Lahan 2"},
}

LAHAN_AKTIF = 1          # <<<<<< HANYA BARIS INI YANG DIGANTI (1 atau 2)

assert LAHAN_AKTIF in LAHAN, f"LAHAN_AKTIF={LAHAN_AKTIF} tidak terdaftar di LAHAN."
NAMA_LAHAN = LAHAN[LAHAN_AKTIF]["nama"]

# ============================================================================
# CONFIG — Satu-satunya titik konfigurasi
# ============================================================================
CONFIG = {
    # -- Input --
    "drone_images_folder":  LAHAN[LAHAN_AKTIF]["folder"],   # diturunkan dari LAHAN_AKTIF

    # -- NodeODM (server stitching lokal) --
    "nodeodm_host":         "localhost",
    "nodeodm_port":         3000,
    "nodeodm_token":        "",                    # Kosong jika belum set token
    "task_name":            f"rice_paddy_lahan{LAHAN_AKTIF}",   # Nama task di NodeODM

    # -- Opsi ODM --
    "odm_options": {
        "orthophoto-resolution":    5,             # cm per piksel orthophoto
        "feature-quality":          "medium",      # low / medium / high / ultra
        "fast-orthophoto":          True,          # Skip 3D model = lebih cepat
        "auto-boundary":            True,          # Otomatis crop area kosong
        "min-num-features":         8000,
    },

    # -- Threshold HSV (skala OpenCV) --
    "hsv_lower_yellow": [20,  60,  60],            # Kuning bawah  → padi MATANG
    "hsv_upper_yellow": [34, 255, 255],            # Kuning atas
    "hsv_lower_green":  [35,  40,  40],            # Hijau bawah   → padi BELUM matang
    "hsv_upper_green":  [85, 255, 255],            # Hijau atas

    # -- Klasifikasi 2 kelas --
    "threshold_siap_panen": 0.60,                  # ≥ 60% kuning → "Siap Panen"

    # -- Folder output --
    "output_folder":        "output",

    # -- Parameter pra-pemrosesan --
    "gaussian_kernel":      5,
    "clahe_clip_limit":     2.0,
    "clahe_tile_size":      8,
    "morph_kernel":         5,

    # -- Performance --
    "max_visualize_size":   2000,                  # Downsample untuk PNG dokumentasi (px)
    "max_web_png_size":     4000,                  # Downsample untuk PNG web Leaflet (px) — sama seperti rice_classification_api
    "polling_interval":     10,                    # Detik antar polling status ODM
    "max_wait_minutes":     120,                   # Timeout maksimum (2 jam)
}

# Validasi
assert 0 < CONFIG["threshold_siap_panen"] < 1, \
    "❌ threshold_siap_panen harus antara 0 dan 1 (misal 0.60)."

# Timestamp sesi untuk file output
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"✅ Konfigurasi siap. Timestamp sesi: {TIMESTAMP}")
print(f"   LAHAN AKTIF         : {LAHAN_AKTIF} ({NAMA_LAHAN})")
print(f"   Folder foto drone   : {CONFIG['drone_images_folder']}")
print(f"   NodeODM             : http://{CONFIG['nodeodm_host']}:{CONFIG['nodeodm_port']}")
print(f"   Threshold siap panen: {CONFIG['threshold_siap_panen']*100:.0f}% piksel kuning")
print(f"   Struktur output     : {CONFIG['output_folder']}/<kategori>/<file>.*")
print(f"                         kategori: orthophoto, orthophoto_web, classification,")
print(f"                                   classification_web, visualization, results")
print(f"                         (sama seperti struktur output rice_classification_api)")

## Sel 4 — Cek Prasyarat Sistem

1. **Docker** terpasang & berjalan
2. **NodeODM container** sudah aktif dan menerima koneksi di `localhost:3000`

In [ ]:
# ============================================================================
# Cek Docker & NodeODM
# ============================================================================
def cek_docker():
    """Cek apakah Docker terpasang & service-nya jalan."""
    try:
        hasil = subprocess.run(
            ["docker", "--version"],
            capture_output=True, text=True, timeout=5
        )
        if hasil.returncode == 0:
            return True, hasil.stdout.strip()
        return False, hasil.stderr.strip()
    except FileNotFoundError:
        return False, "Perintah 'docker' tidak ditemukan di PATH."
    except subprocess.TimeoutExpired:
        return False, "Docker tidak merespons (Docker Desktop mungkin belum jalan)."
    except Exception as e:
        return False, str(e)


def cek_nodeodm(host, port, token=""):
    """Cek apakah NodeODM container responsif via API."""
    try:
        node = Node(host, port, token=token, timeout=10)
        info = node.info()
        return True, info
    except NodeConnectionError as e:
        return False, f"Tidak bisa konek ke NodeODM: {e}"
    except Exception as e:
        return False, f"Error: {e}"


# Cek Docker
print("🔍 Memeriksa Docker...")
ok_docker, info_docker = cek_docker()
if ok_docker:
    print(f"   ✅ {info_docker}")
else:
    print(f"   ❌ {info_docker}")
    print()
    print("   SOLUSI:")
    print("   1. Pastikan Docker Desktop terpasang dari https://www.docker.com/products/docker-desktop")
    print("   2. Buka Docker Desktop, tunggu sampai status 'Running' (icon hijau)")
    print("   3. Restart sel ini")
    raise RuntimeError("Docker belum siap. Lihat instruksi di atas.")

# Cek NodeODM
print()
print(f"🔍 Memeriksa NodeODM di {CONFIG['nodeodm_host']}:{CONFIG['nodeodm_port']}...")
ok_node, info_node = cek_nodeodm(
    CONFIG["nodeodm_host"],
    CONFIG["nodeodm_port"],
    CONFIG["nodeodm_token"],
)
if ok_node:
    print(f"   ✅ NodeODM aktif")
    print(f"      Versi          : {info_node.version}")
    print(f"      Engine         : {info_node.engine} {info_node.engine_version}")
    print(f"      Task slots     : {info_node.max_parallel_tasks} parallel max")
    print(f"      Task running   : {info_node.task_queue_count}")
else:
    print(f"   ❌ {info_node}")
    print()
    print("   SOLUSI:")
    print("   1. Buka Command Prompt / PowerShell di folder proyek")
    print("   2. Jalankan: start_nodeodm.bat")
    print("      Atau manual: docker run -d -p 3000:3000 --name nodeodm opendronemap/nodeodm")
    print("   3. Tunggu 30–60 detik (download image saat pertama kali)")
    print("   4. Restart sel ini")
    raise RuntimeError("NodeODM belum siap. Lihat instruksi di atas.")

print()
print("✅ Semua prasyarat terpenuhi. Lanjut ke Sel 5.")

## Sel 5 — Pemuatan & Validasi Foto Drone

Memeriksa folder foto drone:
- Apakah ada foto JPG/JPEG
- Berapa banyak foto (ODM butuh **minimum 4 foto** untuk stitching)
- Apakah foto punya metadata **EXIF GPS** (penting untuk georeferensi orthophoto otomatis)
- Menampilkan thumbnail grid agar Anda yakin foto yang akan diproses adalah yang benar

In [ ]:
# ============================================================================
# Load & validasi foto drone
# ============================================================================
folder_drone = Path(CONFIG["drone_images_folder"])
if not folder_drone.exists():
    raise FileNotFoundError(
        f"❌ Folder foto drone tidak ditemukan: {folder_drone.resolve()}\n"
        f"   Buat folder tersebut dan letakkan foto drone (JPG) di dalamnya."
    )

# Cari semua foto JPG/JPEG (ODM merekomendasikan JPG, bukan TIF)
ext_valid = {".jpg", ".jpeg", ".JPG", ".JPEG"}
foto_paths = sorted([p for p in folder_drone.iterdir()
                     if p.suffix in ext_valid and p.is_file()])

if len(foto_paths) < 4:
    raise ValueError(
        f"❌ Hanya ditemukan {len(foto_paths)} foto. ODM membutuhkan minimum 4 foto.\n"
        f"   Folder: {folder_drone.resolve()}"
    )

print(f"📷 Ditemukan {len(foto_paths)} foto drone di {folder_drone}")

# Cek EXIF GPS pada subset foto (sampling untuk hemat waktu)
import exifread
def punya_gps(path):
    try:
        with open(path, "rb") as f:
            tags = exifread.process_file(f, details=False, stop_tag="GPS GPSLongitude")
        return "GPS GPSLatitude" in tags and "GPS GPSLongitude" in tags
    except Exception:
        return False

sample_n = min(5, len(foto_paths))
with_gps = sum(punya_gps(p) for p in foto_paths[:sample_n])
print(f"   EXIF GPS check : {with_gps}/{sample_n} foto sampel memiliki tag GPS")
if with_gps == 0:
    print("   ⚠️  PERINGATAN: foto tidak memiliki EXIF GPS.")
    print("       ODM masih bisa stitch, tapi orthophoto TIDAK akan ber-georeferensi.")
    print("       Untuk visualisasi GIS yang benar, gunakan foto dengan EXIF GPS aktif.")

# Tampilkan thumbnail grid (maksimum 12 foto pertama)
n_show = min(12, len(foto_paths))
rows = (n_show + 3) // 4
fig, axes = plt.subplots(rows, 4, figsize=(14, 3.2 * rows))
axes = np.atleast_2d(axes).flatten() if rows > 1 else axes
if not isinstance(axes, np.ndarray):
    axes = [axes]
for i in range(n_show):
    img = cv2.imread(str(foto_paths[i]))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    axes[i].set_title(f"{foto_paths[i].name}", fontsize=8)
    axes[i].axis("off")
for j in range(n_show, len(axes)):
    axes[j].axis("off")
plt.suptitle(f"Preview {n_show} dari {len(foto_paths)} foto drone",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Simpan list path untuk dipakai sel berikutnya
foto_paths_str = [str(p) for p in foto_paths]
print(f"\n✅ Validasi selesai. {len(foto_paths)} foto siap di-stitch.")

## Sel 6 — Stitching Orthophoto dengan ODM

Mengirim foto ke NodeODM, menjalankan stitching, lalu mengunduh orthophoto hasil.

**Catatan waktu pemrosesan:** sangat bervariasi tergantung jumlah foto dan spesifikasi laptop:

| Jumlah foto | RAM 8 GB | RAM 16 GB |
|---|---|---|
| 10–20 | 5–15 menit | 3–8 menit |
| 30–50 | 15–40 menit | 10–25 menit |
| 100+ | 1–3 jam | 30 menit – 1.5 jam |

Tetap di sel ini sampai status menampilkan ✅ — jangan jalankan sel berikutnya selama proses berlangsung.

In [ ]:
# ============================================================================
# Submit task ke NodeODM, polling status, download hasil
# ============================================================================
node = Node(CONFIG["nodeodm_host"], CONFIG["nodeodm_port"],
            token=CONFIG["nodeodm_token"])

print(f"📤 Mengirim {len(foto_paths_str)} foto ke NodeODM...")
t_mulai = time.time()

try:
    task = node.create_task(
        files=foto_paths_str,
        options=CONFIG["odm_options"],
        name=f"{CONFIG['task_name']}_{TIMESTAMP}",
    )
    print(f"   ✅ Task dibuat (UUID: {task.uuid})")
except Exception as e:
    raise RuntimeError(f"❌ Gagal membuat task ODM: {e}")

# Polling status sampai selesai atau gagal
print(f"\n⏳ Menunggu ODM menyelesaikan stitching (polling tiap {CONFIG['polling_interval']} detik)...")
print(f"   Timeout maksimum   : {CONFIG['max_wait_minutes']} menit")
print(f"   Anda dapat membuka http://{CONFIG['nodeodm_host']}:{CONFIG['nodeodm_port']} di browser")
print(f"   untuk melihat progress detail.")
print()

batas_waktu = t_mulai + CONFIG["max_wait_minutes"] * 60
progress_terakhir = -1

with tqdm(total=100, desc="ODM Progress", unit="%", bar_format="{l_bar}{bar}| {n_fmt}%") as pbar:
    while True:
        info = task.info()
        status_kode = info.status.value if hasattr(info.status, "value") else info.status

        # Update progress bar
        progress_baru = int(info.progress) if info.progress else 0
        if progress_baru > progress_terakhir:
            pbar.update(progress_baru - progress_terakhir)
            progress_terakhir = progress_baru

        # Status: 10=QUEUED, 20=RUNNING, 30=FAILED, 40=COMPLETED, 50=CANCELED
        if status_kode == 40:
            pbar.update(100 - progress_terakhir)
            break
        elif status_kode == 30:
            # Coba dapatkan output error
            try:
                log_lines = task.output(line=max(0, info.processing_time // 100 - 30))
                log_text = "\n".join(log_lines[-30:]) if log_lines else "(log kosong)"
            except Exception:
                log_text = "(tidak bisa baca log)"
            raise RuntimeError(
                f"❌ ODM task GAGAL.\n\n"
                f"30 baris log terakhir:\n{log_text}\n\n"
                f"Penyebab umum: foto kurang overlap, tidak ada EXIF GPS, "
                f"RAM tidak cukup. Coba turunkan feature-quality ke 'low' di CONFIG."
            )
        elif status_kode == 50:
            raise RuntimeError("❌ ODM task DIBATALKAN.")

        # Cek timeout
        if time.time() > batas_waktu:
            task.cancel()
            raise TimeoutError(
                f"❌ Timeout {CONFIG['max_wait_minutes']} menit terlampaui. "
                f"Task dibatalkan."
            )

        time.sleep(CONFIG["polling_interval"])

t_selesai = time.time()
durasi_menit = (t_selesai - t_mulai) / 60
print(f"\n✅ Stitching selesai dalam {durasi_menit:.1f} menit.")

# Download hasil
print("\n📥 Mengunduh hasil...")
out_folder = Path(CONFIG["output_folder"]) / "orthophoto"
out_folder.mkdir(parents=True, exist_ok=True)
task_folder = out_folder / f"task_{TIMESTAMP}"
task_folder.mkdir(exist_ok=True)

try:
    download_folder = task.download_assets(str(task_folder))
    print(f"   ✅ Hasil diunduh ke: {download_folder}")
except PermissionError as e:
    # Windows: ZIP sudah ter-extract tapi tidak bisa dihapus (ditahan antivirus/indexer)
    # Ekstraksi sudah berhasil, cukup lanjutkan
    if any(task_folder.rglob("odm_orthophoto.tif")):
        download_folder = str(task_folder)
        print(f"   ✅ Hasil diunduh ke: {download_folder}")
        print(f"   ⚠️  File ZIP sementara tidak bisa dihapus otomatis (normal di Windows, bisa dihapus manual)")
    else:
        raise RuntimeError(f"❌ Gagal mengunduh hasil: {e}")
except Exception as e:
    raise RuntimeError(f"❌ Gagal mengunduh hasil: {e}")

# Cari orthophoto utama: odm_orthophoto/odm_orthophoto.tif
ortho_path = Path(download_folder) / "odm_orthophoto" / "odm_orthophoto.tif"
if not ortho_path.exists():
    # Fallback: scan recursively
    candidates = list(Path(download_folder).rglob("odm_orthophoto.tif"))
    if not candidates:
        raise FileNotFoundError(
            f"❌ orthophoto.tif tidak ditemukan di {download_folder}.\n"
            f"   Cek isi folder secara manual."
        )
    ortho_path = candidates[0]

# Copy ke lokasi standar dengan timestamp
ortho_final = out_folder / f"orthophoto_{TIMESTAMP}.tif"
shutil.copy2(ortho_path, ortho_final)
print(f"\n✅ Orthophoto tersedia di: {ortho_final}")
print(f"   Ukuran file: {ortho_final.stat().st_size / 1024 / 1024:.1f} MB")

## Sel 7 — Pemuatan Orthophoto & Analisis Metadata

Memuat orthophoto GeoTIFF dengan rasterio, membaca metadata geospasial (CRS, bounds, resolusi), dan menampilkan preview. Orthophoto biasanya berukuran besar — sel ini juga menghitung versi *downsampled* untuk visualisasi cepat.

In [ ]:
# ============================================================================
# Load orthophoto & analisis metadata
# ============================================================================
print(f"📂 Memuat orthophoto: {ortho_final.name}")
with rasterio.open(ortho_final) as src:
    n_bands = src.count
    w, h = src.width, src.height
    crs = src.crs
    bounds = src.bounds
    transform = src.transform
    res_x, res_y = src.res

    print(f"   Dimensi      : {w} × {h} piksel")
    print(f"   Jumlah band  : {n_bands} (RGB{'A' if n_bands == 4 else ''})")
    print(f"   CRS          : {crs}")
    print(f"   Bounds       : {bounds}")
    print(f"   Resolusi     : {res_x:.3f} × {res_y:.3f} unit/piksel")

    # Baca semua band
    arr = src.read()                              # (bands, h, w)
    img_full = reshape_as_image(arr)              # (h, w, bands)

    # Deteksi NoData mask:
    # - Jika ada alpha band (band 4), pakai itu
    # - Kalau tidak, deteksi piksel di mana semua RGB = 0 (hitam = ODM nodata default)
    if n_bands == 4:
        mask_valid = img_full[:, :, 3] > 0
        img_rgb = img_full[:, :, :3]
        print(f"   Mask valid   : dari alpha band")
    else:
        img_rgb = img_full[:, :, :3] if n_bands >= 3 else img_full
        mask_valid = np.any(img_rgb > 0, axis=2)
        print(f"   Mask valid   : deteksi dari piksel non-hitam")

    n_valid = int(mask_valid.sum())
    pct_valid = n_valid / mask_valid.size * 100
    print(f"   Piksel valid : {n_valid:,} ({pct_valid:.1f}% dari total)")
    print(f"   Piksel NoData: {mask_valid.size - n_valid:,}")

# Downsample untuk display (tidak mempengaruhi data asli)
max_size = CONFIG["max_visualize_size"]
if max(w, h) > max_size:
    scale = max_size / max(w, h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    img_display = cv2.resize(img_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)
    mask_display = cv2.resize(mask_valid.astype(np.uint8), (new_w, new_h),
                              interpolation=cv2.INTER_NEAREST).astype(bool)
    print(f"   Display      : di-downsample ke {new_w} × {new_h} untuk visualisasi")
else:
    img_display = img_rgb
    mask_display = mask_valid

# Tampilkan orthophoto + mask valid
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(img_display)
axes[0].set_title("Orthophoto Hasil ODM", fontsize=12, fontweight="bold")
axes[0].axis("off")
axes[1].imshow(mask_display, cmap="gray")
axes[1].set_title(f"Mask Area Valid (putih)\n{pct_valid:.1f}% dari total piksel",
                  fontsize=12, fontweight="bold")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"\n✅ Orthophoto siap diproses. Lanjut ke pra-pemrosesan.")

## Sel 8 — Pra-pemrosesan Orthophoto

1. **Gaussian Blur** — reduksi noise sensor
2. **CLAHE pada kanal V (HSV)** — meratakan distribusi pencahayaan tanpa merusak warna (Hue & Saturation tetap utuh)
3. **Konversi BGR → HSV** — untuk segmentasi

> **Catatan**: piksel NoData/transparan **dihindari** dari semua operasi agar tidak mengganggu statistik HSV. Mask validitas dari Sel 7 (`mask_valid`) dipakai sepanjang pipeline.

In [ ]:
# ============================================================================
# Pra-pemrosesan dengan menghindari piksel NoData
# ============================================================================
# Penanda waktu awal tahap KLASIFIKASI (dipakai Tabel Waktu Pemrosesan di BAB 4).
# Waktu stitching diukur terpisah di Sel 6, jadi angka ini murni tahap analisis
# setelah orthophoto tersedia, sesuai kriteria keberhasilan yang diuji.
T_KLASIFIKASI_MULAI = time.time()

# Untuk operasi OpenCV kita perlu BGR (bukan RGB)
img_bgr = cv2.cvtColor(img_rgb.astype(np.uint8), cv2.COLOR_RGB2BGR)

# Langkah 1: Gaussian Blur
k = CONFIG["gaussian_kernel"]
if k % 2 == 0: k += 1
img_blur = cv2.GaussianBlur(img_bgr, (k, k), sigmaX=0)

# Langkah 2: konversi ke HSV → CLAHE pada V
img_hsv_pre = cv2.cvtColor(img_blur, cv2.COLOR_BGR2HSV)
h_ch, s_ch, v_ch = cv2.split(img_hsv_pre)

clahe = cv2.createCLAHE(
    clipLimit=CONFIG["clahe_clip_limit"],
    tileGridSize=(CONFIG["clahe_tile_size"], CONFIG["clahe_tile_size"]),
)
v_clahe = clahe.apply(v_ch)
image_hsv = cv2.merge([h_ch, s_ch, v_clahe])

# Untuk display: kembali ke RGB & downsample
img_after_bgr = cv2.cvtColor(image_hsv, cv2.COLOR_HSV2BGR)
img_after_rgb = cv2.cvtColor(img_after_bgr, cv2.COLOR_BGR2RGB)

if max(w, h) > CONFIG["max_visualize_size"]:
    img_after_display = cv2.resize(img_after_rgb, (img_display.shape[1], img_display.shape[0]),
                                   interpolation=cv2.INTER_AREA)
else:
    img_after_display = img_after_rgb

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(img_display)
axes[0].set_title("Sebelum Pra-pemrosesan", fontsize=11, fontweight="bold")
axes[0].axis("off")
axes[1].imshow(img_after_display)
axes[1].set_title("Setelah Pra-pemrosesan\n(Gaussian Blur + CLAHE pada V)",
                  fontsize=11, fontweight="bold")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"✅ Pra-pemrosesan selesai.")
print(f"   Output: image_hsv siap untuk segmentasi (shape: {image_hsv.shape})")

## Sel 9 — Segmentasi HSV 2 Kelas

Menghasilkan dua mask:
- **`mask_yellow`** — piksel padi matang (Hue ≈ kuning keemasan)
- **`mask_green`** — piksel padi belum matang (Hue ≈ hijau)

Kedua mask dibatasi pada area valid orthophoto (`mask_valid`) sehingga NoData tidak ikut terhitung. Operasi morfologi (Opening + Closing) dibersihkan noise.

Statistik global yang dihitung:
- Total piksel padi (kuning + hijau)
- Persentase kuning relatif terhadap total padi → **dasar verdict** klasifikasi 2 kelas

In [ ]:
# ============================================================================
# Segmentasi HSV — 2 kelas dengan exclusion NoData
# ============================================================================
lower_yellow = np.array(CONFIG["hsv_lower_yellow"], dtype=np.uint8)
upper_yellow = np.array(CONFIG["hsv_upper_yellow"], dtype=np.uint8)
lower_green  = np.array(CONFIG["hsv_lower_green"],  dtype=np.uint8)
upper_green  = np.array(CONFIG["hsv_upper_green"],  dtype=np.uint8)

mask_yellow_raw = cv2.inRange(image_hsv, lower_yellow, upper_yellow)
mask_green_raw  = cv2.inRange(image_hsv, lower_green,  upper_green)

# CRUCIAL: buang piksel NoData dari kedua mask
mask_valid_u8 = (mask_valid.astype(np.uint8) * 255)
mask_yellow_raw = cv2.bitwise_and(mask_yellow_raw, mask_valid_u8)
mask_green_raw  = cv2.bitwise_and(mask_green_raw,  mask_valid_u8)

# Operasi morfologi pada kedua mask
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (CONFIG["morph_kernel"], CONFIG["morph_kernel"]),
)
mask_yellow = cv2.morphologyEx(mask_yellow_raw, cv2.MORPH_OPEN,  kernel)
mask_yellow = cv2.morphologyEx(mask_yellow,     cv2.MORPH_CLOSE, kernel)
mask_green  = cv2.morphologyEx(mask_green_raw,  cv2.MORPH_OPEN,  kernel)
mask_green  = cv2.morphologyEx(mask_green,      cv2.MORPH_CLOSE, kernel)

# Pastikan hasil morfologi tidak merembes keluar area valid
mask_yellow = cv2.bitwise_and(mask_yellow, mask_valid_u8)
mask_green  = cv2.bitwise_and(mask_green,  mask_valid_u8)

# ============================================================================
# RESOLUSI IRISAN PASCA-MORFOLOGI
# ----------------------------------------------------------------------------
# Rentang Hue kuning dan hijau sudah saling lepas, sehingga keluaran cv2.inRange
# dijamin tidak beririsan. Namun operasi morfologi Closing bersifat melebarkan
# (dilasi), sehingga tepi mask kuning dapat melebar ke wilayah mask hijau dan
# sebaliknya. Akibatnya sejumlah kecil piksel dapat masuk ke KEDUA mask, lalu
# terhitung dua kali pada n_yellow + n_green.
#
# Piksel yang diperebutkan diselesaikan berdasarkan nilai Hue aslinya, bukan
# berdasarkan urutan operasi. Piksel dengan Hue di dalam rentang kuning tetap
# menjadi kuning, sisanya menjadi hijau. Dengan cara ini ketentuan perancangan
# "setiap piksel padi terklasifikasi ke dalam tepat satu kelas" benar benar
# terpenuhi, bukan sekadar diasumsikan.
# ============================================================================
irisan = cv2.bitwise_and(mask_yellow, mask_green)
n_irisan_pra = int(np.sum(irisan > 0))

if n_irisan_pra > 0:
    hue_asli   = image_hsv[:, :, 0]
    milik_kuning = (irisan > 0) & (hue_asli <= CONFIG["hsv_upper_yellow"][0])
    milik_hijau  = (irisan > 0) & (hue_asli >  CONFIG["hsv_upper_yellow"][0])
    mask_green[milik_kuning]  = 0
    mask_yellow[milik_hijau]  = 0

n_irisan_pasca = int(np.sum(cv2.bitwise_and(mask_yellow, mask_green) > 0))
print(f"Irisan mask pasca-morfologi : {n_irisan_pra:,} piksel -> "
      f"diselesaikan menjadi {n_irisan_pasca:,} piksel")
assert n_irisan_pasca == 0, "Resolusi irisan gagal. Periksa nilai CONFIG hsv_upper_yellow."

# Statistik
n_valid_total = int(mask_valid.sum())
n_yellow      = int(np.sum(mask_yellow > 0))
n_green       = int(np.sum(mask_green  > 0))
n_rice        = n_yellow + n_green                     # total piksel padi

if n_rice == 0:
    raise ValueError(
        "❌ Tidak ada piksel padi yang terdeteksi.\n"
        "   Kemungkinan threshold HSV tidak cocok dengan citra ini.\n"
        "   Sesuaikan CONFIG['hsv_*'] dan jalankan ulang."
    )

pct_yellow = n_yellow / n_rice * 100
pct_green  = n_green  / n_rice * 100

# Visualisasi
def downsample_mask(m, target_h, target_w):
    if m.shape != (target_h, target_w):
        return cv2.resize(m, (target_w, target_h), interpolation=cv2.INTER_NEAREST)
    return m

th = img_display.shape[0]
tw = img_display.shape[1]
my_disp = downsample_mask(mask_yellow, th, tw)
mg_disp = downsample_mask(mask_green,  th, tw)

# Gabungan untuk visualisasi
combined = np.zeros((th, tw, 3), dtype=np.uint8)
combined[my_disp > 0] = [244, 196, 48]
combined[mg_disp > 0] = [76, 175, 80]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes[0, 0].imshow(img_display); axes[0, 0].set_title("Orthophoto Asli", fontsize=11, fontweight="bold"); axes[0, 0].axis("off")
axes[0, 1].imshow(my_disp, cmap="YlOrBr"); axes[0, 1].set_title(f"Mask KUNING (matang)\n{n_yellow:,} piksel", fontsize=11, fontweight="bold"); axes[0, 1].axis("off")
axes[1, 0].imshow(mg_disp, cmap="Greens");  axes[1, 0].set_title(f"Mask HIJAU (belum matang)\n{n_green:,} piksel", fontsize=11, fontweight="bold"); axes[1, 0].axis("off")
axes[1, 1].imshow(combined);                axes[1, 1].set_title("Gabungan Hasil Segmentasi", fontsize=11, fontweight="bold"); axes[1, 1].axis("off")
plt.tight_layout()
plt.show()

print(f"✅ Segmentasi HSV selesai.")
print(f"   Area valid       : {n_valid_total:,} piksel")
print(f"   Total piksel padi: {n_rice:,}  ({n_rice/n_valid_total*100:.1f}% dari area valid)")
print(f"   Piksel KUNING    : {n_yellow:,}  ({pct_yellow:.2f}% dari padi)")
print(f"   Piksel HIJAU     : {n_green:,}   ({pct_green:.2f}% dari padi)")

if n_rice / n_valid_total < 0.10:
    print()
    print("   ⚠️  PERINGATAN: total piksel padi < 10% dari area valid.")
    print("       Kemungkinan threshold HSV tidak cocok atau lahan tidak didominasi padi.")

## Sel 9b — Uji Sensitivitas Batas Hue (Sel Diagnostik)

Sel ini **tidak** menghasilkan output pipeline. Fungsinya menguji seberapa peka hasil
klasifikasi terhadap pemilihan batas Hue antara kelas kuning dan hijau, serta
membuktikan secara empiris bahwa rentang lama (kuning 20-40, hijau 35-85) memang
beririsan di Hue 35-40.

**Konvensi penamaan:** seluruh variabel di sel ini berawalan `uji_`. Aturan ini wajib
dipertahankan. Pada sesi sebelumnya sel ini memakai nama `n_rice` dan `verdict`
sehingga menimpa variabel pipeline utama dan mengotori JSON serta CSV hasil.

Ambang Saturation dan Value diambil dari `CONFIG`, bukan di-hardcode, supaya
perbandingan antar skenario benar-benar hanya menguji batas Hue.


In [ ]:
# ============================================================================
# SEL DIAGNOSTIK / EKSPERIMEN — Uji sensitivitas batas Hue
# ----------------------------------------------------------------------------
# ATURAN: sel ini HANYA untuk analisis, tidak menghasilkan output pipeline.
# Semua variabel diberi awalan `uji_` supaya TIDAK menimpa variabel pipeline
# utama (n_yellow, n_green, n_rice, pct_yellow, verdict, ...). Jangan hapus
# awalan tersebut. Riwayat: pada sesi sebelumnya loop di sel ini menimpa
# variabel global `n_rice` dan `verdict`, sehingga JSON/CSV ikut tercemar.
# ============================================================================
def uji_klasifikasi_dengan_potong(image_hsv, mask_valid,
                                  potong_kuning_maks, potong_hijau_min,
                                  kernel_size):
    """Hitung ulang segmentasi dengan batas Hue alternatif.

    Ambang Saturation dan Value diambil langsung dari CONFIG (tidak di-hardcode)
    supaya hasil uji setara dengan pipeline produksi. Yang divariasikan HANYA
    batas Hue, sehingga selisih angka murni berasal dari batas Hue.
    """
    uji_s_min_y, uji_v_min_y = CONFIG["hsv_lower_yellow"][1], CONFIG["hsv_lower_yellow"][2]
    uji_s_min_g, uji_v_min_g = CONFIG["hsv_lower_green"][1],  CONFIG["hsv_lower_green"][2]

    uji_lower_y = np.array([CONFIG["hsv_lower_yellow"][0], uji_s_min_y, uji_v_min_y], dtype=np.uint8)
    uji_upper_y = np.array([potong_kuning_maks, 255, 255], dtype=np.uint8)
    uji_lower_g = np.array([potong_hijau_min, uji_s_min_g, uji_v_min_g], dtype=np.uint8)
    uji_upper_g = np.array([CONFIG["hsv_upper_green"][0], 255, 255], dtype=np.uint8)

    uji_mask_y = cv2.inRange(image_hsv, uji_lower_y, uji_upper_y)
    uji_mask_g = cv2.inRange(image_hsv, uji_lower_g, uji_upper_g)

    uji_mv = (mask_valid.astype(np.uint8) * 255)
    uji_mask_y = cv2.bitwise_and(uji_mask_y, uji_mv)
    uji_mask_g = cv2.bitwise_and(uji_mask_g, uji_mv)

    uji_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    for uji_op in (cv2.MORPH_OPEN, cv2.MORPH_CLOSE):
        uji_mask_y = cv2.morphologyEx(uji_mask_y, uji_op, uji_kernel)
        uji_mask_g = cv2.morphologyEx(uji_mask_g, uji_op, uji_kernel)

    # jaga-jaga: morfologi tidak boleh merembes keluar area valid
    uji_mask_y = cv2.bitwise_and(uji_mask_y, uji_mv)
    uji_mask_g = cv2.bitwise_and(uji_mask_g, uji_mv)

    uji_ny    = int(np.sum(uji_mask_y > 0))
    uji_ng    = int(np.sum(uji_mask_g > 0))
    uji_nrice = uji_ny + uji_ng
    uji_pct   = uji_ny / uji_nrice * 100 if uji_nrice else 0.0
    # jumlah piksel yang masuk KEDUA mask -> harus 0 kalau rentang tidak beririsan
    uji_overlap = int(np.sum((uji_mask_y > 0) & (uji_mask_g > 0)))
    return uji_ny, uji_ng, uji_nrice, uji_pct, uji_overlap


uji_kernel_asli   = CONFIG["morph_kernel"]
uji_thr_pct       = CONFIG["threshold_siap_panen"] * 100
uji_potong_produksi = (CONFIG["hsv_upper_yellow"][0], CONFIG["hsv_lower_green"][0])

uji_skenario = [
    ("Overlap lama    (40/35)", 40, 35),   # kondisi bug: Hue 35-40 dihitung 2x
    ("Condong kuning  (40/41)", 40, 41),
    ("Potong tengah   (37/38)", 37, 38),
    (f"PRODUKSI        ({uji_potong_produksi[0]}/{uji_potong_produksi[1]})",
     uji_potong_produksi[0], uji_potong_produksi[1]),
]

print(f"Ambang S/V diambil dari CONFIG -> kuning S>={CONFIG['hsv_lower_yellow'][1]}, "
      f"V>={CONFIG['hsv_lower_yellow'][2]} | hijau S>={CONFIG['hsv_lower_green'][1]}, "
      f"V>={CONFIG['hsv_lower_green'][2]}")
print()
print(f"{'Skenario':26s} {'Kuning':>12s} {'Hijau':>12s} {'Rice':>12s} {'%Kuning':>9s} {'Overlap':>10s}  Verdict")
print("-" * 100)
uji_hasil = []
for uji_nama, uji_py, uji_ph in uji_skenario:
    uji_ny, uji_ng, uji_nrice, uji_pct, uji_ovl = uji_klasifikasi_dengan_potong(
        image_hsv, mask_valid, uji_py, uji_ph, uji_kernel_asli
    )
    uji_verdict = "SIAP PANEN" if uji_pct >= uji_thr_pct else "BELUM SIAP PANEN"
    uji_hasil.append({
        "skenario": uji_nama.strip(), "hue_kuning_maks": uji_py, "hue_hijau_min": uji_ph,
        "n_kuning": uji_ny, "n_hijau": uji_ng, "n_rice": uji_nrice,
        "pct_kuning": round(uji_pct, 2), "n_overlap": uji_ovl, "verdict": uji_verdict,
    })
    print(f"{uji_nama:26s} {uji_ny:>12,} {uji_ng:>12,} {uji_nrice:>12,} "
          f"{uji_pct:8.2f}% {uji_ovl:>10,}  {uji_verdict}")

print()
print("Kolom Overlap = jumlah piksel yang masuk mask kuning DAN hijau sekaligus.")
print("Nilainya harus 0 untuk rentang yang tidak beririsan. Pada skenario")
print("'Overlap lama (40/35)' nilainya > 0, dan itulah sumber bug lama yang bikin")
print("rice_pct_of_valid pernah tembus di atas 100 persen karena satu piksel")
print("terhitung dua kali di penyebut maupun pembilang.")
print()
print("Semua variabel di sel ini berawalan 'uji_' sehingga aman dijalankan")
print("tanpa merusak variabel pipeline utama.")


## Sel 9c — Histogram Sebaran Hue (Sel Diagnostik)

Histogram ini menjadi dasar empiris pemilihan batas Hue pada pendekatan
*rentang indikatif yang dikalibrasi*. Sel ini juga menghitung berapa banyak piksel
yang berada di zona irisan lama (Hue 35 sampai 40), yaitu piksel yang dahulu
terhitung dua kali.

**Konvensi penamaan:** kanal HSV di sel ini memakai `hue_ch`, `sat_ch`, `val_ch`
agar tidak bertabrakan dengan `h_ch`, `s_ch`, `v_ch` pada Sel 8 maupun variabel
`h` dan `w` (dimensi orthophoto) pada Sel 7. Variabel bantu lain berawalan `uji_`.


In [ ]:
# ============================================================================
# SEL DIAGNOSTIK / EKSPERIMEN — Sebaran Hue piksel kandidat padi
# ----------------------------------------------------------------------------
# ATURAN: kanal HSV di sel ini dinamai hue_ch / sat_ch / val_ch supaya tidak
# menimpa h_ch / s_ch / v_ch milik Sel 8, dan tidak menimpa variabel `h`, `w`
# (dimensi orthophoto) dari Sel 7. Variabel bantu lain berawalan `uji_`.
# ============================================================================
hue_ch = image_hsv[:, :, 0]
sat_ch = image_hsv[:, :, 1]
val_ch = image_hsv[:, :, 2]

# Kandidat padi memakai ambang S/V paling permisif di antara kedua kelas, supaya
# histogram memperlihatkan sebaran penuh dan tidak terpotong lebih dulu.
uji_s_floor = min(CONFIG["hsv_lower_yellow"][1], CONFIG["hsv_lower_green"][1])
uji_v_floor = min(CONFIG["hsv_lower_yellow"][2], CONFIG["hsv_lower_green"][2])
uji_hue_min = CONFIG["hsv_lower_yellow"][0]
uji_hue_max = CONFIG["hsv_upper_green"][0] + 5

uji_kandidat = (
    mask_valid
    & (hue_ch >= uji_hue_min) & (hue_ch <= uji_hue_max)
    & (sat_ch >= uji_s_floor) & (val_ch >= uji_v_floor)
)
uji_hue_padi = hue_ch[uji_kandidat]

uji_batas_kuning = CONFIG["hsv_upper_yellow"][0]   # batas atas kuning (produksi)
uji_batas_hijau  = CONFIG["hsv_lower_green"][0]    # batas bawah hijau (produksi)

plt.figure(figsize=(10, 5))
plt.hist(uji_hue_padi, bins=range(uji_hue_min, uji_hue_max + 1),
         color="#6b8e23", edgecolor="white")
plt.axvline(uji_batas_kuning + 0.5, color="orange", linestyle="--",
            label=f"batas atas kuning = {uji_batas_kuning} (produksi)")
plt.axvline(uji_batas_hijau, color="green", linestyle="--",
            label=f"batas bawah hijau = {uji_batas_hijau} (produksi)")
plt.axvline(40.5, color="red", linestyle=":",
            label="batas kuning lama = 40 (beririsan, sudah ditinggalkan)")
plt.xlabel("Nilai Hue (skala OpenCV 0-179)")
plt.ylabel("Jumlah piksel")
plt.title(f"Sebaran Hue piksel kandidat padi (S>={uji_s_floor}, V>={uji_v_floor})")
plt.legend(fontsize=9)
plt.tight_layout()

uji_hist_folder = Path(CONFIG["output_folder"]) / "diagnostik"
uji_hist_folder.mkdir(parents=True, exist_ok=True)
uji_hist_path = uji_hist_folder / f"histogram_hue_{ortho_final.stem}.png"
plt.savefig(uji_hist_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Histogram disimpan ke: {uji_hist_path}")
print()

print(f"Jumlah piksel kandidat padi: {uji_hue_padi.size:,}")
print("Persentil Hue piksel kandidat padi:")
for uji_p in [5, 10, 25, 50, 75, 90, 95]:
    print(f"  P{uji_p:<3d}: {np.percentile(uji_hue_padi, uji_p):.1f}")

uji_n_zona_irisan = int(np.sum((uji_hue_padi >= uji_batas_hijau) & (uji_hue_padi <= 40)))
print()
print(f"Piksel pada zona irisan lama (Hue {uji_batas_hijau}-40): "
      f"{uji_n_zona_irisan:,} "
      f"({uji_n_zona_irisan / max(uji_hue_padi.size, 1) * 100:.2f}% dari kandidat)")
print("Inilah besaran piksel yang dahulu terhitung dua kali. Angka ini dipakai")
print("sebagai bukti kuantitatif di pembahasan BAB 4.")


## Sel 9d — Uji Ambang Saturation dan Value (Sel Diagnostik)

Batas Hue dikunci pada nilai produksi (kuning sampai `hsv_upper_yellow[0]`, hijau mulai
`hsv_lower_green[0]`). Yang divariasikan di sini **hanya ambang S dan V**, supaya terlihat
seberapa besar pengaruh ambang tersebut terhadap jumlah piksel dan verdict.

Tiga kombinasi yang diuji:

| Kombinasi | Kuning | Hijau | Sifat |
|---|---|---|---|
| A. Simetris longgar | S,V >= 40 | S,V >= 40 | pembeda kelas murni Hue |
| B. Simetris ketat | S,V >= 60 | S,V >= 60 | pembeda kelas murni Hue, lebih tahan bayangan |
| C. Asimetris (kondisi CONFIG saat ini) | S,V >= 60 | S,V >= 40 | ada bias struktural |

Kolom **Kuning hilang** menghitung piksel yang Hue-nya masuk rentang kuning tetapi
tertolak oleh ambang S atau V, sehingga tidak masuk kelas mana pun. Kolom inilah bukti
kuantitatif bias pada kombinasi C. Pakai angka dari tabel ini untuk menentukan pilihan
final dan untuk menyusun justifikasi di BAB 3.

Seluruh variabel berawalan `uji_sv_` sehingga tidak menimpa variabel pipeline.


In [ ]:
# ============================================================================
# SEL DIAGNOSTIK / EKSPERIMEN — Uji ambang Saturation dan Value
# ----------------------------------------------------------------------------
# Batas Hue DIKUNCI pada nilai produksi. Yang divariasikan hanya S dan V.
# Semua variabel berawalan `uji_sv_` agar tidak menimpa variabel pipeline.
# ============================================================================
def uji_sv_hitung(image_hsv, mask_valid, s_min_y, v_min_y, s_min_g, v_min_g, kernel_size):
    """Segmentasi ulang dengan ambang S/V tertentu, batas Hue tetap dari CONFIG."""
    uji_sv_hue_y_lo = CONFIG["hsv_lower_yellow"][0]
    uji_sv_hue_y_hi = CONFIG["hsv_upper_yellow"][0]
    uji_sv_hue_g_lo = CONFIG["hsv_lower_green"][0]
    uji_sv_hue_g_hi = CONFIG["hsv_upper_green"][0]

    uji_sv_my = cv2.inRange(image_hsv,
                            np.array([uji_sv_hue_y_lo, s_min_y, v_min_y], dtype=np.uint8),
                            np.array([uji_sv_hue_y_hi, 255, 255], dtype=np.uint8))
    uji_sv_mg = cv2.inRange(image_hsv,
                            np.array([uji_sv_hue_g_lo, s_min_g, v_min_g], dtype=np.uint8),
                            np.array([uji_sv_hue_g_hi, 255, 255], dtype=np.uint8))

    # piksel ber-Hue kuning TANPA syarat S/V, untuk mengukur berapa yang hilang
    uji_sv_my_hue_saja = cv2.inRange(image_hsv,
                                     np.array([uji_sv_hue_y_lo, 0, 0], dtype=np.uint8),
                                     np.array([uji_sv_hue_y_hi, 255, 255], dtype=np.uint8))

    uji_sv_mv = (mask_valid.astype(np.uint8) * 255)
    uji_sv_my = cv2.bitwise_and(uji_sv_my, uji_sv_mv)
    uji_sv_mg = cv2.bitwise_and(uji_sv_mg, uji_sv_mv)
    uji_sv_my_hue_saja = cv2.bitwise_and(uji_sv_my_hue_saja, uji_sv_mv)

    uji_sv_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    for uji_sv_op in (cv2.MORPH_OPEN, cv2.MORPH_CLOSE):
        uji_sv_my = cv2.morphologyEx(uji_sv_my, uji_sv_op, uji_sv_kernel)
        uji_sv_mg = cv2.morphologyEx(uji_sv_mg, uji_sv_op, uji_sv_kernel)
    uji_sv_my = cv2.bitwise_and(uji_sv_my, uji_sv_mv)
    uji_sv_mg = cv2.bitwise_and(uji_sv_mg, uji_sv_mv)

    uji_sv_ny    = int(np.sum(uji_sv_my > 0))
    uji_sv_ng    = int(np.sum(uji_sv_mg > 0))
    uji_sv_nrice = uji_sv_ny + uji_sv_ng
    uji_sv_pct   = uji_sv_ny / uji_sv_nrice * 100 if uji_sv_nrice else 0.0
    uji_sv_hilang = int(np.sum(uji_sv_my_hue_saja > 0)) - int(np.sum(
        cv2.bitwise_and(uji_sv_my_hue_saja,
                        cv2.inRange(image_hsv,
                                    np.array([uji_sv_hue_y_lo, s_min_y, v_min_y], dtype=np.uint8),
                                    np.array([uji_sv_hue_y_hi, 255, 255], dtype=np.uint8))) > 0))
    uji_sv_ovl = int(np.sum((uji_sv_my > 0) & (uji_sv_mg > 0)))
    return uji_sv_ny, uji_sv_ng, uji_sv_nrice, uji_sv_pct, uji_sv_hilang, uji_sv_ovl


uji_sv_thr_pct = CONFIG["threshold_siap_panen"] * 100
uji_sv_kernel_asli = CONFIG["morph_kernel"]

uji_sv_kombinasi = [
    ("A. Simetris longgar  (40/40 | 40/40)", 40, 40, 40, 40),
    ("B. Simetris ketat    (60/60 | 60/60)", 60, 60, 60, 60),
    ("C. Asimetris CONFIG  (60/60 | 40/40)",
     CONFIG["hsv_lower_yellow"][1], CONFIG["hsv_lower_yellow"][2],
     CONFIG["hsv_lower_green"][1],  CONFIG["hsv_lower_green"][2]),
]

print(f"Batas Hue dikunci: kuning {CONFIG['hsv_lower_yellow'][0]}-{CONFIG['hsv_upper_yellow'][0]}, "
      f"hijau {CONFIG['hsv_lower_green'][0]}-{CONFIG['hsv_upper_green'][0]}")
print(f"Area valid orthophoto: {int(mask_valid.sum()):,} piksel")
print()
print(f"{'Kombinasi':38s} {'Kuning':>11s} {'Hijau':>11s} {'Rice':>11s} "
      f"{'%Kuning':>8s} {'%Rice/valid':>12s} {'Kuning hilang':>14s}  Verdict")
print("-" * 130)

uji_sv_rekap = []
for uji_sv_nama, uji_sv_sy, uji_sv_vy, uji_sv_sg, uji_sv_vg in uji_sv_kombinasi:
    (uji_sv_ny, uji_sv_ng, uji_sv_nrice,
     uji_sv_pct, uji_sv_hilang, uji_sv_ovl) = uji_sv_hitung(
        image_hsv, mask_valid, uji_sv_sy, uji_sv_vy, uji_sv_sg, uji_sv_vg, uji_sv_kernel_asli)
    uji_sv_pct_valid = uji_sv_nrice / int(mask_valid.sum()) * 100
    uji_sv_verdict = "SIAP PANEN" if uji_sv_pct >= uji_sv_thr_pct else "BELUM SIAP PANEN"
    uji_sv_rekap.append({
        "kombinasi": uji_sv_nama.strip(),
        "s_min_kuning": uji_sv_sy, "v_min_kuning": uji_sv_vy,
        "s_min_hijau": uji_sv_sg, "v_min_hijau": uji_sv_vg,
        "n_kuning": uji_sv_ny, "n_hijau": uji_sv_ng, "n_rice": uji_sv_nrice,
        "pct_kuning": round(uji_sv_pct, 2),
        "rice_pct_of_valid": round(uji_sv_pct_valid, 2),
        "kuning_hilang_karena_sv": uji_sv_hilang,
        "n_overlap": uji_sv_ovl,
        "verdict": uji_sv_verdict,
    })
    print(f"{uji_sv_nama:38s} {uji_sv_ny:>11,} {uji_sv_ng:>11,} {uji_sv_nrice:>11,} "
          f"{uji_sv_pct:7.2f}% {uji_sv_pct_valid:11.2f}% {uji_sv_hilang:>14,}  {uji_sv_verdict}")

print()
print("Cara membaca tabel:")
print("  - Kolom 'Kuning hilang' = piksel ber-Hue kuning yang ditolak ambang S atau V,")
print("    jadi tidak dihitung sebagai kuning maupun hijau. Makin besar, makin kuat")
print("    bias sistem terhadap verdict BELUM SIAP PANEN.")
print("  - Kolom 'Overlap' internal harus 0 di semua kombinasi. Kalau tidak 0, berarti")
print("    batas Hue di CONFIG kembali beririsan.")
print("  - Kalau verdict kolom paling kanan SAMA di ketiga kombinasi, artinya kesimpulan")
print("    lahan ini robust terhadap pemilihan ambang S/V dan itu poin kuat untuk BAB 4.")
print("    Kalau BERBEDA, kamu wajib memilih satu kombinasi dan menjustifikasinya di BAB 3.")

uji_sv_folder = Path(CONFIG["output_folder"]) / "diagnostik"
uji_sv_folder.mkdir(parents=True, exist_ok=True)
uji_sv_csv = uji_sv_folder / f"uji_ambang_sv_lahan{LAHAN_AKTIF}.csv"
pd.DataFrame(uji_sv_rekap).to_csv(uji_sv_csv, index=False, encoding="utf-8")
print()
print(f"Rekap disimpan ke: {uji_sv_csv}")
print("Jalankan sel ini untuk KEDUA lahan, lalu kirim kedua tabel ini kepadaku.")


## Sel 10 — Klasifikasi Whole-Orthophoto (2 Kelas)
Klasifikasi:

$$
\text{verdict} = \begin{cases}
\text{Siap Panen} & \text{jika } \frac{\text{piksel kuning}}{\text{piksel padi}} \geq \text{threshold} \\
\text{Belum Siap Panen} & \text{lainnya}
\end{cases}
$$

Dengan `threshold = ` `CONFIG["threshold_siap_panen"]`.


In [ ]:
# ============================================================================
# GUARD INTEGRITAS VARIABEL  (wajib jalan sebelum apa pun diekspor)
# ----------------------------------------------------------------------------
# Mencegah terulangnya kasus sesi lalu: sel diagnostik menimpa variabel global
# sehingga JSON/CSV berisi angka yang tidak konsisten (kuning + hijau != rice)
# dan rice_pct_of_valid ikut salah. Kalau assert di bawah gagal, JANGAN pakai
# CSV-nya. Lakukan Restart Kernel lalu Run All berurutan dari Sel 1.
# ============================================================================
assert n_yellow + n_green == n_rice, (
    "GUARD GAGAL: n_yellow + n_green != n_rice "
    f"({n_yellow:,} + {n_green:,} = {n_yellow + n_green:,} vs n_rice = {n_rice:,}).\n"
    "Variabel pipeline tercemar oleh sel lain. Restart Kernel lalu Run All."
)
assert n_rice <= n_valid_total, (
    f"GUARD GAGAL: n_rice ({n_rice:,}) melebihi n_valid_total ({n_valid_total:,}). "
    "Indikasi piksel terhitung ganda atau mask merembes keluar area valid."
)
assert int(np.sum((mask_yellow > 0) & (mask_green > 0))) == 0, (
    "GUARD GAGAL: mask_yellow dan mask_green masih beririsan setelah tahap "
    "resolusi irisan di Sel 9. Periksa CONFIG['hsv_upper_yellow'][0] dan "
    "CONFIG['hsv_lower_green'][0] supaya tidak tumpang tindih."
)
print(f"[GUARD OK] kuning {n_yellow:,} + hijau {n_green:,} = rice {n_rice:,} "
      f"| rice <= valid ({n_valid_total:,}) | irisan mask akhir = 0 "
      f"(irisan pasca-morfologi {n_irisan_pra:,} px sudah diselesaikan)")

# ============================================================================
# Klasifikasi whole-orthophoto: 2 kelas
# ============================================================================
threshold = CONFIG["threshold_siap_panen"]
threshold_pct = threshold * 100

if pct_yellow / 100 >= threshold:
    verdict = "SIAP PANEN"
    verdict_class = 1
    icon = "[OK]"
else:
    verdict = "BELUM SIAP PANEN"
    verdict_class = 2
    icon = "[X]"

# Confidence margin (% jauh dari threshold)
margin = pct_yellow - threshold_pct

print()
print("=" * 70)
print(f"  ==>  VERDICT KLASIFIKASI: {verdict}  <==")
print("=" * 70)
print()
print(f"   Persentase piksel kuning  : {pct_yellow:.2f}%")
print(f"   Ambang batas siap panen   : {threshold_pct:.2f}%")
print(f"   Margin                    : {margin:+.2f} percentage points")
if abs(margin) < 5:
    print(f"   ⚠️  Margin kecil (< 5 pp) — verdict sensitif terhadap perubahan threshold")
else:
    print(f"   ✅ Margin cukup besar — verdict robust terhadap perubahan threshold")

print()
print("=" * 70)
print("⚠️  DISCLAIMER: Verdict di atas adalah ESTIMASI berbasis indikator visual")
print("    warna kanopi pada orthophoto UAV. BUKAN pengukuran parameter fisiologis")
print("    langsung. Hasil dimaksudkan sebagai pendukung keputusan; inspeksi")
print("    lapangan langsung tetap diperlukan untuk keputusan final.")
print("=" * 70)
print(f"   Lahan                     : {NAMA_LAHAN} (LAHAN_AKTIF={LAHAN_AKTIF})")

# Timestamp klasifikasi (terpisah dari TIMESTAMP stitching Sel 3) — sama seperti
# rice_classification_api/api.py::_classify_orthophoto() yang generate timestamp
# baru saat endpoint /classify dipanggil
CLASS_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Statistik tambahan agar field SAMA dengan
# rice_classification_api/services/classification.py::classify_verdict()
valid_pct_of_total = n_valid_total / mask_valid.size * 100
rice_pct_of_valid  = n_rice / n_valid_total * 100 if n_valid_total > 0 else 0

# Simpan untuk dipakai sel-sel berikutnya
WAKTU_KLASIFIKASI_S = round(time.time() - T_KLASIFIKASI_MULAI, 2)

hasil_klasifikasi = {
    "lahan_aktif":          LAHAN_AKTIF,
    "nama_lahan":           NAMA_LAHAN,
    "timestamp":            CLASS_TIMESTAMP,
    "orthophoto_file":      ortho_final.name,
    "n_drone_images":       len(foto_paths),
    "orthophoto_width":     w,
    "orthophoto_height":    h,
    "crs":                  str(crs),
    "n_pixels_total":       int(mask_valid.size),
    "n_pixels_valid":       n_valid_total,
    "n_pixels_rice":        n_rice,
    "n_pixels_yellow":      n_yellow,
    "n_pixels_green":       n_green,
    "yellow_pct_of_rice":   round(pct_yellow, 2),
    "green_pct_of_rice":    round(pct_green, 2),
    "valid_pct_of_total":   round(valid_pct_of_total, 2),
    "rice_pct_of_valid":    round(rice_pct_of_valid, 2),
    "threshold_used":       threshold,
    "verdict":              verdict,
    "verdict_class":        verdict_class,
    "confidence_margin_pp": round(margin, 2),
    "waktu_klasifikasi_s":  WAKTU_KLASIFIKASI_S,
    "irisan_pasca_morfologi": n_irisan_pra,
    "hsv_kuning":           f"H{CONFIG['hsv_lower_yellow'][0]}-{CONFIG['hsv_upper_yellow'][0]} "
                            f"S>={CONFIG['hsv_lower_yellow'][1]} V>={CONFIG['hsv_lower_yellow'][2]}",
    "hsv_hijau":            f"H{CONFIG['hsv_lower_green'][0]}-{CONFIG['hsv_upper_green'][0]} "
                            f"S>={CONFIG['hsv_lower_green'][1]} V>={CONFIG['hsv_lower_green'][2]}",
}

## Sel 11 — Ekspor GeoTIFF Klasifikasi (Untuk QGIS)

Raster GeoTIFF dengan nilai:

| Nilai | Arti | Warna QGIS |
|---|---|---|
| 0 | NoData / bukan padi | transparan |
| 1 | Padi MATANG (kuning) | kuning |
| 2 | Padi BELUM matang (hijau) | hijau |

GeoTIFF ini mewarisi **CRS dan transform** dari orthophoto sumber, sehingga otomatis sejajar saat ditumpangkan di atas orthophoto atau basemap di QGIS.

**Disamakan dengan `rice_classification_api`**: nama file mengikuti skema
`<stem>_class_<timestamp>.tif` (lihat `services/classification.py::export_classification_geotiff()`
dan `api.py::_classify_orthophoto()`), disimpan ke `output/classification/`.
Sel 11b dan 11c di bawah menambahkan ekspor PNG web (Leaflet) yang sudah ada
di API tapi sebelumnya belum ada di pipeline notebook ini.

In [ ]:
# ============================================================================
# Ekspor GeoTIFF klasifikasi
# ============================================================================
# Build raster klasifikasi
class_raster = np.zeros((h, w), dtype=np.uint8)
class_raster[mask_yellow > 0] = 1                     # Padi matang
class_raster[mask_green  > 0] = 2                     # Padi belum matang
# (NoData & non-padi tetap 0)

# Stem & timestamp klasifikasi — skema penamaan SAMA dengan rice_classification_api
# (lihat api.py::_classify_orthophoto): "<stem>_class_<timestamp>.tif"
stem = ortho_final.stem

out_class_folder = Path(CONFIG["output_folder"]) / "classification"
out_class_folder.mkdir(parents=True, exist_ok=True)
class_path = out_class_folder / f"{stem}_class_{CLASS_TIMESTAMP}.tif"

try:
    with rasterio.open(
        class_path, "w",
        driver="GTiff",
        height=h, width=w, count=1,
        dtype=class_raster.dtype,
        crs=crs, transform=transform,
        compress="lzw", nodata=0,
    ) as dst:
        dst.write(class_raster, 1)
        dst.update_tags(
            1,
            DESCRIPTION="Klasifikasi kematangan padi 2 kelas",
            VALUE_0="NoData / Bukan Padi",
            VALUE_1="Padi Matang (Kuning)",
            VALUE_2="Padi Belum Matang (Hijau)",
            VERDICT_GLOBAL=verdict,
            YELLOW_PCT=f"{pct_yellow:.2f}%",
        )
    size_mb = class_path.stat().st_size / 1024 / 1024
    print(f"✅ GeoTIFF klasifikasi berhasil diekspor.")
    print(f"   Path    : {class_path}")
    print(f"   Ukuran  : {size_mb:.2f} MB")
    print(f"   Dimensi : {w} × {h} (CRS: {crs})")
except Exception as e:
    print(f"❌ Gagal mengekspor GeoTIFF: {e}")

## Sel 11b — Ekspor Orthophoto ke PNG RGBA (Web)

Mengonversi orthophoto GeoTIFF menjadi PNG RGBA: piksel NoData dibuat
transparan agar basemap OpenStreetMap tetap terlihat di luar polygon sawah
(perilaku sama seperti layer GeoTIFF di QGIS). Sel ini juga menghitung
`bounds_wgs84` (lat/lon) yang dipakai `L.imageOverlay()` di Leaflet dan
disimpan ke JSON ringkasan (Sel 13).

Output: `output/orthophoto_web/<stem>_ortho_web_<timestamp>.png`

In [ ]:
# ============================================================================
# Ekspor orthophoto → PNG RGBA untuk Leaflet ImageOverlay
# Identik dengan rice_classification_api/services/classification.py
#   ::export_orthophoto_web_png()
# ============================================================================
from rasterio.warp import transform_bounds
from rasterio.crs import CRS

out_ortho_web_folder = Path(CONFIG["output_folder"]) / "orthophoto_web"
out_ortho_web_folder.mkdir(parents=True, exist_ok=True)
ortho_web_path = out_ortho_web_folder / f"{stem}_ortho_web_{CLASS_TIMESTAMP}.png"

try:
    with rasterio.open(ortho_final) as src:
        arr_web = src.read()
        if arr_web.dtype != np.uint8:
            arr_web = np.clip(arr_web, 0, 255).astype(np.uint8)

        r_ch, g_ch, b_ch = arr_web[0], arr_web[1], arr_web[2]

        # Alpha channel: dari band ke-4 kalau ada, atau deteksi piksel hitam (NoData)
        if n_bands >= 4:
            a_ch = arr_web[3]
        else:
            a_ch = np.where((r_ch == 0) & (g_ch == 0) & (b_ch == 0), 0, 255).astype(np.uint8)

        # OpenCV pakai urutan BGRA (bukan RGBA)
        bgra_ortho = np.stack([b_ch, g_ch, r_ch, a_ch], axis=-1)

        max_web = CONFIG["max_web_png_size"]
        h_web, w_web = bgra_ortho.shape[:2]
        if max(h_web, w_web) > max_web:
            scale_web = max_web / max(h_web, w_web)
            new_w_web, new_h_web = int(w_web * scale_web), int(h_web * scale_web)
            # RGB: INTER_AREA lebih halus untuk downscale
            bgr_down = cv2.resize(bgra_ortho[:, :, :3], (new_w_web, new_h_web), interpolation=cv2.INTER_AREA)
            # Alpha: INTER_NEAREST agar tepi polygon tidak blur
            a_down   = cv2.resize(bgra_ortho[:, :, 3],  (new_w_web, new_h_web), interpolation=cv2.INTER_NEAREST)
            bgra_ortho = np.dstack([bgr_down, a_down])

        # Bounds WGS84 untuk Leaflet L.imageOverlay()
        wgs84 = CRS.from_epsg(4326)
        if crs.to_epsg() == 4326:
            bounds_wgs84 = [bounds.left, bounds.bottom, bounds.right, bounds.top]
        else:
            left, bottom, right, top = transform_bounds(crs, wgs84, *bounds)
            bounds_wgs84 = [left, bottom, right, top]

    cv2.imwrite(str(ortho_web_path), bgra_ortho)
    size_kb = ortho_web_path.stat().st_size / 1024
    print(f"✅ PNG orthophoto web berhasil diekspor.")
    print(f"   Path        : {ortho_web_path}")
    print(f"   Ukuran      : {size_kb:.0f} KB")
    print(f"   Bounds WGS84: {bounds_wgs84}")
except Exception as e:
    print(f"❌ Gagal mengekspor PNG orthophoto web: {e}")

## Sel 11c — Ekspor Overlay Klasifikasi ke PNG RGBA (Web)

Overlay transparan untuk ditumpuk di atas PNG orthophoto (Sel 11b) di Leaflet:

| Kelas | Warna RGBA |
|---|---|
| Bukan padi | `(0,0,0,0)` — transparan total |
| Padi MATANG (kuning) | `(255,235,59,200)` |
| Padi BELUM matang (hijau) | `(76,175,80,200)` |

Output: `output/classification_web/<stem>_class_overlay_<timestamp>.png`

In [ ]:
# ============================================================================
# Ekspor overlay klasifikasi → PNG RGBA untuk Leaflet
# Identik dengan rice_classification_api/services/classification.py
#   ::export_classification_overlay_png()
# ============================================================================
out_class_web_folder = Path(CONFIG["output_folder"]) / "classification_web"
out_class_web_folder.mkdir(parents=True, exist_ok=True)
class_overlay_path = out_class_web_folder / f"{stem}_class_overlay_{CLASS_TIMESTAMP}.png"

try:
    bgra_overlay = np.zeros((h, w, 4), dtype=np.uint8)
    # Kuning: RGB(255,235,59) → BGR(59,235,255), alpha=200
    bgra_overlay[mask_yellow > 0] = [59, 235, 255, 200]
    # Hijau: RGB(76,175,80) → BGR(80,175,76), alpha=200
    bgra_overlay[mask_green  > 0] = [80, 175,  76, 200]

    # Downsample dengan INTER_NEAREST agar batas kelas tidak berbaur
    max_web = CONFIG["max_web_png_size"]
    if max(h, w) > max_web:
        scale_web = max_web / max(h, w)
        new_w_web, new_h_web = int(w * scale_web), int(h * scale_web)
        bgra_overlay = cv2.resize(bgra_overlay, (new_w_web, new_h_web), interpolation=cv2.INTER_NEAREST)

    cv2.imwrite(str(class_overlay_path), bgra_overlay)
    size_kb = class_overlay_path.stat().st_size / 1024
    print(f"✅ PNG overlay klasifikasi web berhasil diekspor.")
    print(f"   Path   : {class_overlay_path}")
    print(f"   Ukuran : {size_kb:.0f} KB")
except Exception as e:
    print(f"❌ Gagal mengekspor PNG overlay klasifikasi: {e}")

## Sel 12 — Ekspor PNG Peta Visual (Untuk Dokumentasi Skripsi)

PNG resolusi 300 DPI dengan:
- Orthophoto sebagai *background*
- Overlay semi-transparan area kuning (#FFEB3B) dan hijau (#4CAF50)
- **Verdict besar** di pojok kiri-atas (pakai banner berwarna sesuai status)
- Legenda warna & informasi statistik di pojok kanan-bawah

Cocok langsung untuk lampiran BAB 4.

**Disamakan dengan `rice_classification_api`**: nama file mengikuti skema
`<stem>_map_<timestamp>.png` (lihat `services/classification.py::export_visualization_png()`),
disimpan ke `output/visualization/`.

In [ ]:
# ============================================================================
# Ekspor PNG visualisasi untuk dokumen skripsi
# ============================================================================
fig, ax = plt.subplots(figsize=(15, 11), dpi=200)
ax.imshow(img_display)

# Overlay mask warna semi-transparan
overlay = np.zeros((*img_display.shape[:2], 4), dtype=np.uint8)
overlay[my_disp > 0] = [255, 235, 59, 130]        # kuning + alpha
overlay[mg_disp > 0] = [76, 175, 80, 130]         # hijau + alpha
ax.imshow(overlay)

# Banner verdict di pojok kiri-atas
banner_color = "#2d6a4f" if verdict == "SIAP PANEN" else "#c1121f"
banner_text  = f"  {verdict}  "
ax.text(0.02, 0.97, banner_text,
        transform=ax.transAxes,
        fontsize=20, fontweight="bold", color="white",
        verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.7",
                  facecolor=banner_color, edgecolor="black", linewidth=1.5))

# Info box di pojok kanan-bawah
info_text = (
    f"Persentase kuning : {pct_yellow:.1f}%\n"
    f"Persentase hijau  : {pct_green:.1f}%\n"
    f"Ambang siap panen : {threshold_pct:.0f}%\n"
    f"Margin            : {margin:+.1f} pp\n"
    f"Foto drone        : {len(foto_paths)}\n"
    f"Diproses          : {datetime.now().strftime('%Y-%m-%d %H:%M')}"
)
ax.text(0.98, 0.03, info_text,
        transform=ax.transAxes,
        fontsize=10, family="monospace", color="black",
        verticalalignment="bottom", horizontalalignment="right",
        bbox=dict(boxstyle="round,pad=0.5",
                  facecolor="white", alpha=0.92, edgecolor="gray"))

# Legenda warna
legend_patches = [
    Patch(facecolor=(1.0, 0.92, 0.23, 0.5), edgecolor="black",
          label=f"Padi MATANG (kuning)"),
    Patch(facecolor=(0.30, 0.69, 0.31, 0.5), edgecolor="black",
          label=f"Padi BELUM matang (hijau)"),
]
ax.legend(handles=legend_patches, loc="lower left", framealpha=0.92, fontsize=10)

ax.set_title(f"Peta Kesiapan Panen — {ortho_final.stem}",
             fontsize=13, fontweight="bold", pad=10)
ax.axis("off")

# Simpan — nama file disamakan dengan rice_classification_api: "<stem>_map_<timestamp>.png"
out_viz_folder = Path(CONFIG["output_folder"]) / "visualization"
out_viz_folder.mkdir(parents=True, exist_ok=True)
png_path = out_viz_folder / f"{stem}_map_{CLASS_TIMESTAMP}.png"

try:
    plt.tight_layout()
    plt.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    size_kb = png_path.stat().st_size / 1024
    print(f"✅ PNG peta visual berhasil diekspor.")
    print(f"   Path   : {png_path}")
    print(f"   Ukuran : {size_kb:.0f} KB (300 DPI)")
except Exception as e:
    print(f"❌ Gagal mengekspor PNG: {e}")

## Sel 13 — Ekspor JSON & CSV Ringkasan

In [ ]:
# ============================================================================
# Ekspor JSON & CSV
# ============================================================================
out_results = Path(CONFIG["output_folder"]) / "results"
out_results.mkdir(parents=True, exist_ok=True)


def _to_rel(p: Path) -> str:
    """Convert path ke relative dari folder kerja notebook (sama seperti
    rice_classification_api/api.py::_to_rel(), tapi base-nya Path.cwd())."""
    try:
        return str(Path(p).resolve().relative_to(Path.cwd().resolve()))
    except ValueError:
        return str(Path(p).resolve())


# JSON ringkasan — skema SAMA dengan rice_classification_api/api.py::_classify_orthophoto()
# (full_result): hasil_klasifikasi (= **stats) + bounds + bounds_wgs84 + files{}
full_result = {
    **hasil_klasifikasi,
    "bounds": list(bounds),
    "bounds_wgs84": bounds_wgs84,
    "files": {
        "orthophoto":                 _to_rel(ortho_final),
        "classification":             _to_rel(class_path),
        "orthophoto_web_png":         _to_rel(ortho_web_path),
        "classification_overlay_png": _to_rel(class_overlay_path),
        "visualization":              _to_rel(png_path),
    },
}

# JSON detail — nama file disamakan dengan API: "<stem>_result_<timestamp>.json"
json_path = out_results / f"{stem}_result_{CLASS_TIMESTAMP}.json"
try:
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(full_result, f, indent=2, ensure_ascii=False)
    print(f"✅ JSON ringkasan: {json_path}")
except Exception as e:
    print(f"❌ Gagal menulis JSON: {e}")

# CSV (1 baris untuk eksperimen ini) — fitur khusus notebook, schema flat dari hasil_klasifikasi
csv_path = out_results / f"{stem}_result_{CLASS_TIMESTAMP}.csv"
try:
    df = pd.DataFrame([hasil_klasifikasi])
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"✅ CSV ringkasan: {csv_path}")
except Exception as e:
    print(f"❌ Gagal menulis CSV: {e}")

# CSV master (append) — akumulasi banyak eksperimen, fitur khusus notebook (tidak ada di API)
csv_master = out_results / "master_results.csv"
try:
    df_new = pd.DataFrame([hasil_klasifikasi])
    if csv_master.exists():
        df_master = pd.read_csv(csv_master)
        df_master = pd.concat([df_master, df_new], ignore_index=True)
    else:
        df_master = df_new
    df_master.to_csv(csv_master, index=False, encoding="utf-8")
    print(f"✅ CSV master (akumulasi): {csv_master}  ({len(df_master)} baris total)")
except Exception as e:
    print(f"⚠️  Tidak bisa update CSV master: {e}")

print("\nIsi JSON:")
print(json.dumps(full_result, indent=2, ensure_ascii=False))

## Sel 14 — Database

In [ ]:
# Notebook ada di rice_classification_api/notebooks/, jadi folder backend
# (tempat database.py dan crud.py) adalah induk dari folder kerja notebook.
sys.path.append(str(Path.cwd().resolve().parent))

import database
import crud
from sqlmodel import Session

database.init_db()

# nama_lahan diambil dari registry LAHAN di Sel 3, bukan hardcode, supaya
# label di database selalu sinkron dengan folder foto yang diproses.
print(f"Menyimpan hasil untuk: {NAMA_LAHAN} (LAHAN_AKTIF={LAHAN_AKTIF})")

hasil_json = full_result

with Session(database.engine) as session:
    deteksi = crud.simpan_hasil_deteksi(session, hasil_json, nama_lahan=NAMA_LAHAN)
    print(f"Tersimpan ke database  ->  id_deteksi={deteksi.id_deteksi}, id_lahan={deteksi.id_lahan}")
    print(f"Verdict: {deteksi.verdict}  |  Kematangan: {deteksi.persentase_kematangan}%")

## Sel 15 — Panduan Visualisasi QGIS & Status Versi Web

In [ ]:
# ============================================================================
# Cetak panduan QGIS + status versi web
# ============================================================================
output_paths = {
    "Orthophoto (GeoTIFF dari ODM)":         ortho_final,
    "Orthophoto Web (PNG RGBA Leaflet)":     ortho_web_path,
    "Klasifikasi (GeoTIFF 2 kelas)":         class_path,
    "Klasifikasi Web (PNG overlay Leaflet)": class_overlay_path,
    "Peta Visual (PNG 300 DPI)":             png_path,
    "Ringkasan (JSON)":                      json_path,
    "Ringkasan (CSV)":                       csv_path,
    "Master CSV (akumulasi)":                csv_master,
}

panduan = """
================================================================================
              PANDUAN VISUALISASI HASIL DI QGIS
================================================================================

LANGKAH 1 — Buka QGIS, buat proyek baru. Simpan dengan nama "skripsi_padi.qgz".

LANGKAH 2 — Tambah basemap OpenStreetMap:
   • Plugins → Manage and Install Plugins → cari "QuickMapServices" → Install
   • Web → QuickMapServices → Settings → tab "More services" → "Get contributed pack"
   • Web → QuickMapServices → OSM → OSM Standard

LANGKAH 3 — Drag-drop orthophoto.tif ke QGIS
   • Layer akan otomatis ditempatkan di lokasi geografis yang benar
   • Atur opacity ke ~80% via Layer Properties → Symbology

LANGKAH 4 — Drag-drop klasifikasi.tif ke QGIS
   • Layer akan menampilkan area kuning vs hijau
   • Untuk pewarnaan yang benar:
     - Properties → Symbology → Render type: "Paletted/Unique values"
     - Classify (otomatis deteksi 0, 1, 2)
     - Warna: 0 = transparan, 1 = kuning (#FFEB3B), 2 = hijau (#4CAF50)
   • Atur opacity ke ~70% agar orthophoto di bawahnya tetap terlihat

LANGKAH 5 — Print Layout untuk dokumentasi skripsi
   • Project → New Print Layout → "Peta Hasil Skripsi"
   • Add Map → drag area → peta muncul
   • Add Legend, Add Scale Bar, Add North Arrow, Add Label (judul + verdict)
   • Export as PDF atau PNG DPI 300 untuk lampiran BAB 4

================================================================================
                  STATUS: VERSI WEB SUDAH DIBANGUN
================================================================================

Pipeline ini sudah memiliki versi web yang berjalan offline (edge computing)
di laptop GCS, terdiri dari 2 komponen tambahan yang berkomunikasi via HTTP
REST di localhost:

  ┌──────────────────────────┐         ┌──────────────────────────┐
  │   rice-harvest-web/       │         │  rice_classification_api/ │
  │   Next.js + Leaflet       │ ──HTTP─►│  FastAPI backend           │
  │   Port 3001               │  REST   │  Port 8000                 │
  │   - Upload foto drone     │ ◄──────│  - Stitching via NodeODM    │
  │   - Peta dual layer       │  JSON   │  - Klasifikasi HSV          │
  │   - Verdict + statistik   │         │  - Export GeoTIFF + PNG     │
  └──────────────────────────┘         └──────────────────────────┘

Sel 11b dan 11c di notebook ini menduplikasi persis output PNG RGBA web
(orthophoto_web & classification_web) yang dihasilkan API, dan Sel 13
menulis JSON dengan skema field yang identik. Artinya:
   ✅ Stitching ODM         → sama dengan services/stitching.py
   ✅ Klasifikasi HSV       → sama dengan services/classification.py
   ✅ Export GeoTIFF        → skema nama & tag identik dengan API
   ✅ Export PNG web        → identik (Sel 11b/11c, untuk Leaflet ImageOverlay)
   ✅ Export JSON ringkasan → skema field identik (bounds, bounds_wgs84, files)

Catatan: notebook ini tetap berguna untuk eksperimen ad-hoc dan akumulasi
master_results.csv untuk pengujian akurasi BAB 4 — tidak perlu menjalankan
NodeODM + FastAPI + Next.js sekaligus hanya untuk menguji satu orthophoto.

================================================================================
                       FILE OUTPUT YANG DIHASILKAN
================================================================================
"""
print(panduan)
for nama, path in output_paths.items():
    status = "✅" if Path(path).exists() else "⚠️"
    print(f"   {status}  {nama}")
    print(f"        {Path(path).resolve()}")

print()
print("=" * 80)
print("✨ PIPELINE SELESAI. Hasil siap dipakai untuk BAB 4.")
print("=" * 80)
print()
print("Langkah berikutnya untuk BAB 4:")
print("  Penelitian ini memakai DUA lahan uji, sehingga kerangka pengujiannya adalah")
print("  UJI FUNGSIONAL (black-box) ditambah STUDI KASUS per lahan, BUKAN pengujian")
print("  akurasi statistik. Dengan n = 2, confusion matrix beserta Precision, Recall,")
print("  dan F1-Score tidak memiliki makna statistik dan sebaiknya tidak dilaporkan.")
print("  Yang dilaporkan adalah:")
print("    1. Keberjalanan tiap fungsi beserta artefak buktinya (Sel 17).")
print("    2. Kesesuaian verdict tiap lahan terhadap kondisi lapangan (Sel 16).")
print("    3. Kekokohan verdict terhadap perubahan parameter (Sel 9b dan Sel 9d).")
print("    4. Konsistensi internal hasil, dibuktikan lolosnya GUARD di Sel 10.")
print()
print("  Jalankan notebook untuk LAHAN_AKTIF = 1 lalu 2, kemudian jalankan Sel 16")
print("  untuk merangkum keduanya menjadi tabel siap tempel ke BAB 4.")


## Sel 16 — Rekapitulasi Studi Kasus Dua Lahan (untuk BAB 4)

Sel ini membaca `output/results/master_results.csv`, mengambil **hasil terbaru per lahan**,
lalu menyusun tabel siap tempel ke BAB 4. Jalankan sel ini setelah notebook selesai
dijalankan untuk `LAHAN_AKTIF = 1` dan `LAHAN_AKTIF = 2`.

Sel ini juga memeriksa konsistensi internal setiap baris (kuning + hijau harus sama dengan
rice, dan rice tidak boleh melebihi area valid). Baris yang gagal ditandai agar tidak
terpakai di laporan.


In [ ]:
# ============================================================================
# REKAPITULASI STUDI KASUS DUA LAHAN — output siap tempel ke BAB 4
# ============================================================================
rekap_path = Path(CONFIG["output_folder"]) / "results" / "master_results.csv"
if not rekap_path.exists():
    raise FileNotFoundError(
        f"{rekap_path} belum ada. Jalankan notebook penuh untuk LAHAN_AKTIF = 1 dan 2 dulu."
    )

df_all = pd.read_csv(rekap_path)

if "lahan_aktif" not in df_all.columns:
    raise ValueError(
        "master_results.csv masih memakai skema lama tanpa kolom 'lahan_aktif'.\n"
        "Arsipkan berkas tersebut (rename jadi master_results_ARSIP_prabug.csv),\n"
        "lalu jalankan ulang notebook untuk kedua lahan supaya skemanya baru."
    )

# ---- audit konsistensi tiap baris ----
df_all["cek_jumlah"]  = df_all["n_pixels_yellow"] + df_all["n_pixels_green"] == df_all["n_pixels_rice"]
df_all["cek_subset"]  = df_all["n_pixels_rice"] <= df_all["n_pixels_valid"]
df_all["baris_valid"] = df_all["cek_jumlah"] & df_all["cek_subset"]

n_tolak = int((~df_all["baris_valid"]).sum())
print(f"Total baris di master CSV : {len(df_all)}")
print(f"Baris konsisten           : {int(df_all['baris_valid'].sum())}")
print(f"Baris DITOLAK (tercemar)  : {n_tolak}")
if n_tolak:
    print()
    print("Baris berikut TIDAK BOLEH dipakai di BAB 4 karena kuning + hijau tidak sama")
    print("dengan rice, atau rice melebihi area valid:")
    print(df_all.loc[~df_all["baris_valid"],
                     ["timestamp", "nama_lahan", "n_pixels_yellow",
                      "n_pixels_green", "n_pixels_rice", "n_pixels_valid"]].to_string(index=False))
print()

df_ok = df_all[df_all["baris_valid"]].copy()
if df_ok.empty:
    raise ValueError("Tidak ada baris yang konsisten. Restart Kernel lalu Run All untuk kedua lahan.")

# ---- ambil hasil TERBARU per lahan ----
df_ok = df_ok.sort_values("timestamp")
df_final = df_ok.groupby("lahan_aktif", as_index=False).tail(1).sort_values("lahan_aktif")

print("=" * 100)
print("TABEL A — Deskripsi Data Uji (untuk Tabel 4.1)")
print("=" * 100)
kol_a = ["nama_lahan", "orthophoto_file", "n_drone_images",
         "orthophoto_width", "orthophoto_height", "crs"]
print(df_final[kol_a].to_string(index=False))
print()

print("=" * 100)
print("TABEL B — Hasil Klasifikasi Sistem dengan Parameter Final (untuk Tabel 4.7)")
print("=" * 100)
kol_b = ["nama_lahan", "n_pixels_yellow", "n_pixels_green", "n_pixels_rice",
         "yellow_pct_of_rice", "verdict", "confidence_margin_pp"]
print(df_final[kol_b].to_string(index=False))
print()

print("=" * 100)
print("TABEL C — Konsistensi Internal dan Waktu Pemrosesan (untuk Tabel 4.12 dan 4.13)")
print("=" * 100)
kol_c = ["nama_lahan", "n_pixels_valid", "n_pixels_rice", "rice_pct_of_valid",
         "valid_pct_of_total", "waktu_klasifikasi_s", "hsv_kuning", "hsv_hijau"]
print(df_final[kol_c].to_string(index=False))
print()

# ---- narasi studi kasus per lahan ----
print("=" * 100)
print("RINGKASAN NARATIF STUDI KASUS (bahan paragraf pembahasan BAB 4)")
print("=" * 100)
for _, baris in df_final.iterrows():
    kokoh = "LEBAR" if abs(baris["confidence_margin_pp"]) >= 5 else "SEMPIT"
    print(f"\n{baris['nama_lahan']} ({baris['orthophoto_file']})")
    print(f"  Verdict          : {baris['verdict']}")
    print(f"  Kematangan       : {baris['yellow_pct_of_rice']:.2f}% "
          f"(ambang {baris['threshold_used'] * 100:.0f}%)")
    print(f"  Margin           : {baris['confidence_margin_pp']:+.2f} pp -> {kokoh}")
    print(f"  Piksel padi      : {int(baris['n_pixels_rice']):,} dari "
          f"{int(baris['n_pixels_valid']):,} piksel valid "
          f"({baris['rice_pct_of_valid']:.2f}%)")
    print(f"  Waktu klasifikasi: {baris['waktu_klasifikasi_s']:.2f} detik")
    if kokoh == "SEMPIT":
        print("  CATATAN: margin di bawah 5 pp. Lahan ini adalah KASUS BATAS, verdict-nya")
        print("           sensitif terhadap parameter. Wajib dibahas eksplisit di BAB 4.")

# ---- kekokohan lintas lahan ----
print()
print("=" * 100)
print("KEKOKOHAN LINTAS LAHAN")
print("=" * 100)
verdicts = df_final["verdict"].unique().tolist()
if len(verdicts) == 1:
    print(f"Kedua lahan menghasilkan verdict yang sama: {verdicts[0]}.")
    print("KONSEKUENSI PENTING untuk BAB 4: karena tidak ada variasi kelas pada data uji,")
    print("confusion matrix hanya akan terisi pada satu sel. Precision, Recall, dan")
    print("F1-Score menjadi tidak terdefinisi atau menyesatkan, sehingga TIDAK boleh")
    print("dilaporkan. Laporkan kesesuaian verdict per lahan secara kualitatif saja,")
    print("dan cantumkan keterbatasan ini secara jujur di sub-bab keterbatasan.")
else:
    print(f"Data uji memuat dua kelas verdict yang berbeda: {verdicts}.")
    print("Ini menguatkan argumen bahwa sistem mampu MEMBEDAKAN kondisi lahan, bukan")
    print("selalu menjawab sama. Tetap laporkan sebagai studi kasus, bukan sebagai")
    print("akurasi statistik, karena n = 2 masih terlalu kecil untuk metrik agregat.")

# ---- simpan ----
out_bab4 = Path(CONFIG["output_folder"]) / "results" / "rekap_bab4_dua_lahan.csv"
df_final.drop(columns=["cek_jumlah", "cek_subset", "baris_valid"]).to_csv(
    out_bab4, index=False, encoding="utf-8")
print()
print(f"Rekap tersimpan ke: {out_bab4}")


## Sel 17 — Checklist Uji Fungsional Berbasis Artefak (Black-Box)

Sel ini memverifikasi keberjalanan tiap fungsi dengan memeriksa **artefak nyata** yang
dihasilkan pipeline, bukan sekadar pernyataan bahwa fungsi sudah jalan. Setiap baris
mencantumkan bukti yang dapat ditunjukkan saat sidang, misalnya nama berkas, dimensi
raster, CRS, atau nilai statistik.

Keluarannya berupa CSV yang tinggal ditempel ke tabel hasil pengujian fungsional di BAB 4.


In [ ]:
# ============================================================================
# CHECKLIST UJI FUNGSIONAL — verifikasi berbasis artefak nyata
# ============================================================================
uji_fn = []

def catat(no, fungsi, harapan, lolos, bukti):
    uji_fn.append({
        "no": no, "fungsi_diuji": fungsi, "output_diharapkan": harapan,
        "hasil_aktual": bukti, "status": "BERHASIL" if lolos else "GAGAL",
    })

# 1. Validasi input foto drone
catat(1, "Pemuatan dan validasi foto drone",
      "Foto JPG terbaca, jumlah minimum 4 terpenuhi",
      len(foto_paths) >= 4,
      f"{len(foto_paths)} foto terbaca dari {CONFIG['drone_images_folder']}")

# 2. Stitching ODM
catat(2, "Photogrammetric stitching via NodeODM",
      "Orthophoto GeoTIFF ber-georeferensi tersimpan",
      ortho_final.exists() and crs is not None,
      f"{ortho_final.name}, {w}x{h} px, CRS {crs}, "
      f"{ortho_final.stat().st_size / 1024**2:.1f} MB")

# 3. Deteksi area valid (NoData exclusion)
catat(3, "Deteksi mask area valid",
      "Piksel NoData dikecualikan dari analisis",
      0 < n_valid_total < mask_valid.size,
      f"{n_valid_total:,} piksel valid dari {mask_valid.size:,} "
      f"({valid_pct_of_total:.2f}%)")

# 4. Pra-pemrosesan
catat(4, "Pra-pemrosesan (Gaussian Blur, CLAHE pada V, konversi HSV)",
      "Array HSV terbentuk dengan dimensi sesuai orthophoto",
      image_hsv.shape == (h, w, 3) and image_hsv.dtype == np.uint8,
      f"image_hsv shape {image_hsv.shape}, dtype {image_hsv.dtype}, "
      f"rentang H {int(image_hsv[:,:,0].min())}-{int(image_hsv[:,:,0].max())}")

# 5. Segmentasi HSV dua kelas
catat(5, "Segmentasi HSV dua kelas",
      "Dua mask biner terbentuk, saling lepas, di dalam area valid",
      int(np.sum((mask_yellow > 0) & (mask_green > 0))) == 0,
      f"kuning {n_yellow:,} px, hijau {n_green:,} px, irisan akhir 0 px "
      f"(irisan pasca-morfologi {n_irisan_pra:,} px diselesaikan berdasar Hue asli)")

# 6. Konsistensi hitung piksel
catat(6, "Konsistensi perhitungan statistik piksel",
      "Kuning + hijau sama dengan total padi, dan tidak melebihi area valid",
      (n_yellow + n_green == n_rice) and (n_rice <= n_valid_total),
      f"{n_yellow:,} + {n_green:,} = {n_rice:,} <= {n_valid_total:,} "
      f"(rice/valid = {rice_pct_of_valid:.2f}%)")

# 7. Perhitungan persentase dan penentuan verdict
catat(7, "Perhitungan persentase kematangan dan penentuan verdict",
      "Persentase 0 sampai 100 dan verdict sesuai ambang",
      0 <= pct_yellow <= 100 and verdict in ("SIAP PANEN", "BELUM SIAP PANEN"),
      f"{pct_yellow:.2f}% terhadap ambang {threshold*100:.0f}% "
      f"-> {verdict} (margin {margin:+.2f} pp)")

# 8. Ekspor GeoTIFF klasifikasi
catat(8, "Ekspor GeoTIFF klasifikasi untuk SIG",
      "GeoTIFF nilai kelas per piksel dengan CRS yang sama",
      Path(class_path).exists(),
      f"{Path(class_path).name}, {Path(class_path).stat().st_size / 1024**2:.2f} MB")

# 9. Ekspor PNG web (orthophoto dan overlay)
catat(9, "Ekspor PNG RGBA untuk antarmuka web",
      "Dua PNG transparan beserta bounds WGS84",
      Path(ortho_web_path).exists() and Path(class_overlay_path).exists(),
      f"{Path(ortho_web_path).name} dan {Path(class_overlay_path).name}, "
      f"bounds WGS84 {[round(b, 6) for b in bounds_wgs84]}")

# 10. Ekspor visualisasi dokumentasi
catat(10, "Ekspor PNG peta visual 300 DPI",
      "PNG berisi orthophoto, overlay, verdict, dan legenda",
      Path(png_path).exists(),
      f"{Path(png_path).name}, {Path(png_path).stat().st_size / 1024:.0f} KB")

# 11. Ekspor ringkasan JSON dan CSV
catat(11, "Ekspor ringkasan JSON dan CSV",
      "Berkas JSON dan CSV berisi seluruh field statistik",
      Path(json_path).exists() and Path(csv_path).exists(),
      f"{Path(json_path).name} dan {Path(csv_path).name}, "
      f"{len(hasil_klasifikasi)} field")

# 12. Waktu pemrosesan tahap klasifikasi
catat(12, "Waktu pemrosesan tahap klasifikasi",
      "Tidak lebih dari 5 menit per orthophoto",
      WAKTU_KLASIFIKASI_S <= 300,
      f"{WAKTU_KLASIFIKASI_S:.2f} detik "
      f"({WAKTU_KLASIFIKASI_S/60:.2f} menit) untuk {w}x{h} px")

df_uji_fn = pd.DataFrame(uji_fn)
n_lolos = int((df_uji_fn["status"] == "BERHASIL").sum())

print("=" * 110)
print(f"HASIL UJI FUNGSIONAL — {NAMA_LAHAN}   ({n_lolos} dari {len(df_uji_fn)} skenario BERHASIL)")
print("=" * 110)
for _, r in df_uji_fn.iterrows():
    tanda = "[OK]" if r["status"] == "BERHASIL" else "[GAGAL]"
    print(f"\n{tanda} {r['no']:>2}. {r['fungsi_diuji']}")
    print(f"      Diharapkan : {r['output_diharapkan']}")
    print(f"      Aktual     : {r['hasil_aktual']}")

out_fn = Path(CONFIG["output_folder"]) / "results" / f"uji_fungsional_lahan{LAHAN_AKTIF}.csv"
df_uji_fn.to_csv(out_fn, index=False, encoding="utf-8")
print()
print("=" * 110)
print(f"Tersimpan ke: {out_fn}")
if n_lolos < len(df_uji_fn):
    print("ADA SKENARIO GAGAL. Perbaiki dulu sebelum angka ini dipakai di BAB 4.")
else:
    print("Seluruh skenario berhasil. Tabel ini siap ditempel ke BAB 4.")
print()
print("CATATAN JUJUR untuk laporan: skenario 1 sampai 12 di atas menguji jalur notebook.")
print("Fungsi yang khas antarmuka web, yaitu unggah multi-file, polling progres, riwayat")
print("deteksi, dan tampilan dual layer Leaflet, harus diuji terpisah lewat antarmuka web")
print("dan dibuktikan dengan tangkapan layar, karena tidak dijalankan oleh notebook ini.")
